# RAGAS Evaluation using Groq and HuggingFace

## What is RAGAS?
RAGAS (Retrieval Augmented Generation Assessment) is a powerful framework that allows you to evaluate your RAG pipelines without requiring human-annotated ground truth for every question. Instead of relying on traditional metrics like BLEU or ROUGE (which fail if the model uses synonyms), RAGAS uses an **LLM-as-a-Judge** to evaluate semantic accuracy, reasoning, and retrieval quality.

### Core Metrics Evaluated:
1. **Faithfulness**: Did the AI hallucinate? (Measures if the generated answer is strictly derived from the retrieved context).
2. **Answer Relevancy**: Does the answer directly address the user's question, or is it evasive?
3. **Context Precision**: Did the retriever pull the most relevant documents and rank them at the top?
4. **Context Recall**: Did the retriever fetch *all* the necessary information required to answer the question?

---

## Setup Instructions

Before running the code cells, you need API keys for Groq (our fast LLM judge) and HuggingFace (for embedding models).

### 1. Create a Groq Account
- Visit [console.groq.com](https://console.groq.com/) and sign up for a free developer account.
- Navigate to **API Keys** on the left sidebar and click **Create API Key**.
- Copy the generated key immediately (it starts with `gsk_...`).

### 2. Create a HuggingFace Account
- Visit [huggingface.co](https://huggingface.co/) and sign up.
- Navigate to **Settings > Access Tokens**.
- Click **New Token** (Read permission is sufficient) and copy it.

### 3. Set up your `.env` file
In the exact same folder as this Jupyter Notebook, create a file simply named `.env`. Add the following lines to it, replacing the placeholder text with your actual keys:

```text
GROQ_API_KEY=gsk_your_groq_api_key_here
HUGGINGFACE_API_KEY=hf_your_huggingface_token_here
```

Once your `.env` file is saved, proceed to run the code below!

### Step 1: Library Imports
In this cell, we import all the necessary libraries. We are bringing in `python-dotenv` to securely load our API keys, `datasets` to format our test data, and the core `ragas.metrics` classes that will dictate how our LLM judges the pipeline.

In [2]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame

# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

# LangChain Integrations
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

### Step 2: Securely Load Keys & Initialize Models
Here, we initialize the two AI components that RAGAS requires:
1. **The Judge LLM**: We are using Groq's high-speed inference engine running a LLaMA-3 model. We set the `temperature=0` to ensure the judge is strictly analytical and deterministic.
2. **The Embeddings Model**: We are using HuggingFace's `all-MiniLM-L6-v2`. This model will convert our text into vector numbers so RAGAS can mathematically calculate semantic similarities.

In [3]:
# Load the environment variables from your .env file
load_dotenv(override=True)

# Verify the key is loaded
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# Initialize the LLM Judge via Groq
judge_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=4096
)

# Initialize Local Embeddings via HuggingFace
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model_name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Judge LLM configured as: llama-3.1-8b-instant
✅ Embeddings configured as: sentence-transformers/all-MiniLM-L6-v2


### Step 3: Define the Evaluation Dataset
RAGAS expects your data in a specific format (a HuggingFace `Dataset` object). The standard evaluation requires:
- `question`: The user's original query.
- `answer`: The text generated by your RAG pipeline.
- `contexts`: A list of the document chunks that your Retriever fetched (used to calculate Faithfulness and Precision).
- `ground_truth` (Optional but recommended): The actual right answer (used to calculate Context Recall).

In [5]:
data = {
    'question': ['What is Docker?', 'What is Kubernetes?'],
    'answer': [
        'Docker is a platform for containerizing applications.', 
        'Kubernetes is an orchestration tool for containers.'
    ],
    'contexts' : [
        ['Docker is a set of platform as a service products that use OS-level virtualization to deliver software in packages called containers.'], 
        ['Kubernetes is an open-source container orchestration system for automating software deployment, scaling, and management.']
    ],
    'ground_truth': [
        'Docker is a containerization platform.',
        'Kubernetes automates container deployment and management.'
    ]
}

dataset = Dataset.from_dict(data)

### Step 4: Execute Fast RAGAS Evaluation
Now we pass our dataset, our chosen metrics, and our LLM configurations into the `evaluate` function. 

*Note: Because we are hitting a cloud API (Groq), Ragas will use asyncio to evaluate multiple rows and metrics in parallel, meaning this should run extremely fast compared to a local LLM container.*

In [6]:
print("Initiating RAGAS Evaluation...")
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    raise_exceptions=False,
    batch_size=4
)

# Output as a clean Pandas DataFrame for analysis
results_df = score.to_pandas()
display(results_df)

Initiating RAGAS Evaluation...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Batch 1/2:   0%|          | 0/4 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000015C72EE1F40> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000015C72EE1F40> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-68' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Sujeet\showcase\course_lab\week1\venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Tas

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Docker?,[Docker is a set of platform as a service prod...,Docker is a platform for containerizing applic...,Docker is a containerization platform.,1.0,0.918468,1.0,0.5
1,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an orchestration tool for contai...,Kubernetes automates container deployment and ...,1.0,1.000000,1.0,1.0


## How to Interpret the Output
Once the evaluation completes, look at the scores generated in your DataFrame. Each metric produces a score between **0.0** and **1.0**.

- **High Context Recall + Low Faithfulness**: The retriever found the right information, but your generator hallucinated or ignored the context. Check your generator prompt!
- **Low Context Precision + High Faithfulness**: The generator gave a good answer, but the retriever is pulling a lot of useless garbage documents into the context window. Your embeddings model or chunking strategy might need tuning.
- **Low Answer Relevancy**: The AI gave a true statement, but it dodged the actual question being asked.